# Theme 9: Stereo Vision and 3D Reconstruction

**Objective:** In this exercise set, we try out different methods for extracting depth / 3D information from sets of images.


## Question 1: SSD Block Matching

**Theory:**
Sum of Squared Differences (SSD) is used to find the best match of a template (search block) within a reference image.
$$ SSD(x,y) = \sum_{i,j \in W} (I_1(i,j) - I_2(x+i, y+j))^2 $$
where $I_1$ is the search block ($N 	imes N$) and $I_2$ is the reference image.

**Problem:**
Consider the following sample matrices (since original values were missing):
$I_1$ (Search Block, 2x2):
[[10, 20],
 [30, 40]]

$I_2$ (Reference, 3x3):
[[12, 25, 20],
 [28, 42, 40],
 [30, 90, 100]]

**Questions:**
1.  **Search block size:** 2x2 ($N=2$).
2.  **Valid positions:** For a $H 	imes W$ reference and $N 	imes N$ block, valid top-left $(x,y)$ range is $0 \le x \le W-N$ and $0 \le y \le H-N$.
    *   Here $3 	imes 3$ reference, $2 	imes 2$ block. Range: $x \in [0, 1], y \in [0, 1]$.
3.  **Number of SSD values:** $(W-N+1) 	imes (H-N+1) = 2 	imes 2 = 4$ values.
4.  **Calculate SSD sum:** See code below.
5.  **Minimum SSD:** See code below.


In [ ]:
import numpy as np

# Define matrices (Example values)
I1 = np.array([[10, 20],
               [30, 40]])

I2 = np.array([[12, 25, 20],
               [28, 42, 40],
               [30, 90, 100]])

print("Search Block I1 (2x2):\n", I1)
print("Reference I2 (3x3):\n", I2)

H, W = I2.shape
N = I1.shape[0]

# Calculate SSD for valid positions
min_ssd = float('inf')
best_pos = (0, 0)

print("\nCalculating SSDs:")
for y in range(H - N + 1):
    for x in range(W - N + 1):
        # Extract patch from I2
        patch = I2[y:y+N, x:x+N]
        
        # Calculate SSD
        diff = I1 - patch
        ssd = np.sum(diff**2)
        
        print(f"Pos (x={x}, y={y}): SSD = {ssd}")
        
        if ssd < min_ssd:
            min_ssd = ssd
            best_pos = (x, y)

print(f"\nMinimum SSD is {min_ssd} at position (x={best_pos[0]}, y={best_pos[1]})")


## Question 2: Stereo Disparity

**Task:** Load `left_1.png` and `right_1.png`, perform Stereo Matching, and calculate depth.

**Theory:**
*   **Disparity:** The shift in pixel position of the same object between left and right images. $d = x_L - x_R$.
*   **Depth ($z$):** Inversely proportional to disparity.
    $$ z = rac{B \cdot f}{d} $$
    Where $B$ is baseline (0.2m) and $f$ is focal length (723px).

**Code Instructions:**
*   Adjust `numDisparities` (must be divisible by 16) and `blockSize` (must be odd, usually 5-21) to optimize the disparity map.
*   Larger `numDisparities` allows detecting objects closer to the camera (larger shift).
*   Larger `blockSize` reduces noise but smooths out fine details.


In [ ]:
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt

# Load images
# Make sure 'resources/left_1.png' exists
imgL = cv.imread('resources/left_1.png', 0)
imgR = cv.imread('resources/right_1.png', 0)

if imgL is None or imgR is None:
    print("Error: Images not found in 'resources/' folder.")
else:
    # 1. StereoBM Setup
    numDisparities = 64  # Range of disparity search (divisible by 16)
    blockSize = 15       # Block size for matching (odd)

    stereo = cv.StereoBM_create(numDisparities=numDisparities, blockSize=blockSize)
    
    # 2. Compute Disparity
    disparity = stereo.compute(imgL, imgR)

    # 3. Visualization
    # Disparity map contains fixed-point values (pixels * 16). 
    # For visualization, we normalize.
    disp_vis = cv.normalize(disparity, None, alpha=0, beta=255, norm_type=cv.NORM_MINMAX, dtype=cv.CV_8U)

    plt.figure(figsize=(10, 5))
    plt.imshow(disp_vis, 'gray')
    plt.title(f'Disparity Map (numDisp={numDisparities}, blkSize={blockSize})')
    plt.axis('off')
    plt.colorbar()
    plt.show()

    # 4. Calculate Depth to Yellow Building (Background)
    # We pick a pixel likely belonging to the building (background -> small disparity)
    # Let's say center-top area.
    # Note: Disparity map from StereoBM is 16 * pixel_disparity
    
    # Let's verify a specific point visually roughly where the wall is (e.g. x=300, y=100)
    # You can change this coordinate to probe different pixels
    py, px = 100, 300 
    
    disp_val_16 = disparity[py, px]
    
    if disp_val_16 <= 0:
        print(f"Invalid disparity at ({px}, {py}). Try another point.")
    else:
        true_disparity = disp_val_16 / 16.0
        
        # Constants
        B = 0.2 # meters
        f = 723 # pixels
        
        depth = (B * f) / true_disparity
        
        print(f"Pixel ({px}, {py}) Disparity: {true_disparity} px")
        print(f"Estimated Distance: {depth:.2f} meters")


## Question 3: Fundamental Matrix Estimation

**Task:** Estimate the Fundamental Matrix $F$ between the two views.

**Theory:**
The Fundamental Matrix $F$ encapsulates the epipolar geometry. For any pair of matching points $x \leftrightarrow x'$, the constraint is:
$$ x'^T F x = 0 $$
It maps a point in one image to a line (epipolar line) in the other image.


In [ ]:
import cv2 as cv
import numpy as np

# Load images
img1 = cv.imread('resources/left_1.png', 0)
img2 = cv.imread('resources/right_1.png', 0)

# 1. Detect Features (SIFT)
sift = cv.SIFT_create()
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

# 2. Match Features (FLANN or BF)
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1, des2, k=2)

# Lowe's Ratio Test
good = []
pts1 = []
pts2 = []

for m, n in matches:
    if m.distance < 0.8 * n.distance:
        good.append(m)
        pts1.append(kp1[m.queryIdx].pt)
        pts2.append(kp2[m.trainIdx].pt)

pts1 = np.int32(pts1)
pts2 = np.int32(pts2)

# 3. Compute Fundamental Matrix (using RANSAC or LMEDS)
F, mask = cv.findFundamentalMat(pts1, pts2, cv.FM_LMEDS)

print("Fundamental Matrix F:\n", F)

# Interpret F:
# It relates the two camera views. If cameras are rectified (stereo),
# F often has a specific form (zeros on diagonal, etc.), but with real data
# and LMEDS it estimates the actual relationship.


## Question 4: COLMAP (Optional)

**Task:** Structure-from-Motion 3D Reconstruction using COLMAP.

**Instructions (for Google Colab):**
This part usually requires the COLMAP binary installed, which is pre-installed or easily easy-installable on Google Colab (Linux).

1.  Upload `images.zip` (containing overlapping photos) to Colab and unzip to `/content/images`.
2.  Install COLMAP: `!sudo apt-get install colmap` (on Colab).
3.  Run the reconstruction pipeline commands:
    *   `colmap feature_extractor ...`
    *   `colmap exhaustive_matcher ...`
    *   `colmap mapper ...`
4.  Download the resulting `.ply` point cloud.
5.  Visualize with Meshlab locally.

**Python Code (Skeleton for reference):**
```python
import os

# Example commands (system calls)
# !colmap feature_extractor --database_path database.db --image_path ./images
# !colmap exhaustive_matcher --database_path database.db
# !mkdir sparse
# !colmap mapper --database_path database.db --image_path ./images --output_path ./sparse
# !colmap model_converter --input_path ./sparse/0 --output_path ./model.ply --output_type PLY
```
